# Trajectory Analysis: A First Look

**REQUIRED DAY 2** (time permitting — see [the Day 2 README](../README.md) for how this fits the schedule)

## When cells are mid-transition, not fixed types

Today's clustering (notebooks 07-08) treats cell types as discrete boxes. Real biology is often continuous — a stem cell differentiating, a T cell activating — where cells fall along a spectrum rather than into clean categories. **Pseudotime** orders cells along an inferred developmental/activation trajectory using only a snapshot of expression, not real time-series data — it's an ordering, built from how gradually or abruptly cells' transcriptomes change from one to the next.

## Today's data: blood cell differentiation

[`sc.datasets.paul15`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.datasets.paul15.html) is a classic myeloid/erythroid differentiation dataset (Paul et al., 2015) — bone marrow progenitor cells caught at different stages of committing to become one of several blood cell types. This is one of scanpy's own standard trajectory tutorial datasets, small and fast enough to run live.

Like the other extra datasets today, this is already fetched once into the shared data folder rather than downloaded fresh by everyone mid-class — point scanpy there first.

In [ ]:
import scanpy as sc
import warnings
warnings.filterwarnings("ignore")

sc.settings.datasetdir = "/tscc/nfs/home/juf009/day2_shared_data/extra_datasets"
## Fill in: adata_traj = sc.datasets.paul15()

adata_traj


## Preprocessing — same steps as this morning, once more

Run the same normalization and dimensionality-reduction steps you've now used three times today: `sc.pp.normalize_total`, `sc.pp.log1p`, `sc.pp.highly_variable_genes` (`n_top_genes=1000` — this dataset has fewer genes measured to begin with), `sc.pp.pca`, `sc.pp.neighbors`, `sc.tl.umap`. By this point in the day you should be able to write these six lines from memory rather than looking each one up again — that repetition is deliberate.

In [ ]:
## Fill in the six-step preprocessing pipeline described above


sc.pl.umap(adata_traj)


## PAGA: a map of how clusters connect

Cluster first, same as before (`sc.tl.leiden`). Then, instead of treating each cluster as an unrelated island, [PAGA](https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.paga.html) (`sc.tl.paga`) estimates how *connected* each pair of clusters is — a cluster undergoing continuous differentiation into another should show a strong PAGA connection between them, while two genuinely unrelated cell types should show close to none. Run `sc.tl.leiden`, then `sc.tl.paga(adata_traj, groups="leiden")`, then plot it with `sc.pl.paga`.

In [ ]:
## Fill in:
## sc.tl.leiden(adata_traj, resolution=1.0)
## sc.tl.paga(adata_traj, groups="leiden")
## sc.pl.paga(adata_traj, color="leiden")




Thick lines between clusters mean scanpy found many cells "in between" them transcriptionally — a candidate continuous transition. Thin or absent lines mean the two clusters look like genuinely separate populations. This is worth comparing against your UMAP: PAGA is telling you something about connectivity that a UMAP layout alone can visually suggest but can't quantify.

## Diffusion pseudotime: ordering cells along the trajectory

PAGA tells you *which* clusters connect. [Diffusion pseudotime](https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.dpt.html) (`sc.tl.dpt`) orders individual *cells* along that structure, once you tell it where the trajectory starts. Pseudotime needs a root: pick one cell from your earliest/most progenitor-like cluster (based on what you know about this dataset being progenitors differentiating outward) and set `adata_traj.uns["iroot"]` to its index before running `sc.tl.diffmap` and `sc.tl.dpt`.

In [ ]:
## Fill in:
## 1) Pick a root cell from your most progenitor-like cluster, e.g.:
##    root_candidates = adata_traj.obs_names[adata_traj.obs["leiden"] == "YOUR_PROGENITOR_CLUSTER"]
##    adata_traj.uns["iroot"] = adata_traj.obs_names.get_loc(root_candidates[0])
## 2) sc.tl.diffmap(adata_traj)
## 3) sc.tl.dpt(adata_traj)


sc.pl.umap(adata_traj, color=["leiden", "dpt_pseudotime"])


Compare the two panels: `leiden` shows discrete clusters, `dpt_pseudotime` shows a continuous value per cell. Do the pseudotime values increase smoothly away from your chosen root, radiating out along the branches you saw in the PAGA graph? If pseudotime jumps around discontinuously rather than forming a smooth gradient, that's a sign the root choice or the underlying structure doesn't support a single clean trajectory — worth stating as a limitation, not silently ignoring.

Pseudotime is an *ordering*, not a *direction* — it can't tell you on its own whether cells are moving from cluster A to cluster B or the reverse. You supplied the direction yourself, by choosing which cluster the root came from. **RNA velocity** (mentioned below) is the technique that estimates directionality from data itself, rather than requiring you to assume it.

## What's beyond today

- **RNA velocity** — uses the ratio of spliced to unspliced transcripts in each cell to estimate the *direction* a cell is moving transcriptionally, giving trajectory inference an actual arrow of time instead of a direction you have to assume by picking a root. Needs spliced/unspliced counts (from `velocyto` or `kb-python`), which today's dataset doesn't have — see [scVelo](https://scvelo.readthedocs.io/).
- **Lineage tracing** — uses an experimental label (genetic barcoding, not just expression) to track which cells are descended from which, when you need ground truth rather than an inference.
- **Batch correction / integration** — you already did this hands-on, in [07_dimensionality_reduction_and_clustering.ipynb](07_dimensionality_reduction_and_clustering.ipynb)'s Harmony section — the same idea applies here the moment a trajectory dataset spans more than one batch or donor.
- **Cell-cell communication** — infers likely signaling interactions between cell types based on co-expression of known ligand-receptor pairs (e.g., CellPhoneDB, squidpy's ligand-receptor analysis).
- **Gene regulatory networks (GRNs)** — infers which transcription factors are likely driving which downstream genes, from expression correlation patterns.

## Further reading

- [Single-cell best practices — Trajectory Analysis](https://www.sc-best-practices.org/)
- [scanpy: PAGA/DPT trajectory tutorial](https://scanpy.readthedocs.io/en/stable/tutorials/trajectories/paga-paul15.html)
- [scVelo (RNA velocity)](https://scvelo.readthedocs.io/)
- [CellPhoneDB (cell-cell communication)](https://www.cellphonedb.org/)
- [Monocle3 / DDRTree](https://cole-trapnell-lab.github.io/monocle3/) — the R/Bioconductor equivalent of this notebook's PAGA/DPT approach, and what BMI710's Week 4 activity used directly.